In [6]:
import numpy as np
import pandas as pd


Reading the dataset

In [7]:
df = pd.read_csv('/kaggle/input/datasets/asoktamang/nlp-dataset/train.csv')
df.head()

,anchor,target,rating,score,context
0,aralkynyl,cholesterol,1d,0.25,C09
1,aralkynyl,aralkyl,1d,0.25,C09
2,aralkynyl,heterocycle,1c,0.25,C09
3,aralkynyl,acyl,1d,0.25,C09
4,aralkynyl,heterocyclic,1c,0.25,C09


studying the dataset

In [8]:
df.describe(include= object)

,anchor,target,rating,context
count,36473,36473,36473,36473
unique,733,29340,10,106
top,component composite coating,composition,0,H01
freq,152,24,7471,2186


checking the null values

In [9]:
df.isna().sum()

anchor     0
target     0
rating     0
score      0
context    0
dtype: int64

checking the duplicate datas in df

In [10]:
df.duplicated().sum()

np.int64(0)

Creating a input feature, combining context, target and the anchor feature

In [11]:
df['input'] = 'TEXT1: ' + df['context'] + '; TEXT2: ' + df['target'] + '; ANC1: ' + df['anchor']

In [12]:
df.head()

,anchor,target,rating,score,context,input
0,aralkynyl,cholesterol,1d,0.25,C09,TEXT1: C09; TEXT2: cholesterol; ANC1: aralkynyl
1,aralkynyl,aralkyl,1d,0.25,C09,TEXT1: C09; TEXT2: aralkyl; ANC1: aralkynyl
2,aralkynyl,heterocycle,1c,0.25,C09,TEXT1: C09; TEXT2: heterocycle; ANC1: aralkynyl
3,aralkynyl,acyl,1d,0.25,C09,TEXT1: C09; TEXT2: acyl; ANC1: aralkynyl
4,aralkynyl,heterocyclic,1c,0.25,C09,TEXT1: C09; TEXT2: heterocyclic; ANC1: aralkynyl


Making a huggingface dataset from ourdataframe

In [13]:
from datasets import Dataset,DatasetDict
ds = Dataset.from_pandas(df)             
ds

Dataset({
    features: ['anchor', 'target', 'rating', 'score', 'context', 'input'],
    num_rows: 36473
})

Taking a pre-trained model


In [14]:
model_nm = 'microsoft/deberta-v3-small'

Creation of tokenizer

In [15]:
from transformers import AutoTokenizer
tokz = AutoTokenizer.from_pretrained(model_nm)
tokz

DebertaV2Tokenizer(name_or_path='microsoft/deberta-v3-small', vocab_size=128000, model_max_length=1000000000000000019884624838656, padding_side='right', truncation_side='right', special_tokens={'bos_token': '[CLS]', 'eos_token': '[SEP]', 'unk_token': '[UNK]', 'sep_token': '[SEP]', 'pad_token': '[PAD]', 'cls_token': '[CLS]', 'mask_token': '[MASK]'}, added_tokens_decoder={
	0: AddedToken("[PAD]", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
	1: AddedToken("[CLS]", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
	2: AddedToken("[SEP]", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
	3: AddedToken("[UNK]", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
	128000: AddedToken("[MASK]", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
}
)

In [16]:
def tokenize_data(x):
    return tokz(x['input'])  #tokenizing the input feature of a dataframe

In [17]:
tok_ds = ds.map(tokenize_data,batched=True)

Map:   0%|          | 0/36473 [00:00<?, ? examples/s]